In [1]:
# Mount Google Drive to access project files and saved models.
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import warnings

import numpy as np
import pandas as pd
import torch
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup
)
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support
)

warnings.filterwarnings("ignore")


# ============================================================
# DEVICE SETUP
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Device: {device}")

if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(
        f"GPU Memory: "
        f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB"
    )


# ============================================================
# PATHS
# ============================================================

AUGMENTED_DATASET_DIR = (
    "/content/drive/MyDrive/code_switch_project/data/augmented_dataset"
)

MODEL_OUTPUT_DIR = (
    "/content/drive/MyDrive/code_switch_project/"
    "models/xlm_roberta_binary_codeswitching"
)

RESULTS_OUTPUT_DIR = (
    "/content/drive/MyDrive/code_switch_project/results"
)

os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)
os.makedirs(RESULTS_OUTPUT_DIR, exist_ok=True)


# ============================================================
# FINE-TUNING PARAMETERS
# ============================================================

MODEL_NAME = "xlm-roberta-base"
MAX_LENGTH = 128
BATCH_SIZE = 64
LEARNING_RATE = 2e-5
NUM_EPOCHS = 3
WARMUP_STEPS = 500
WEIGHT_DECAY = 0.01
SEED = 42


# Reproducibility
torch.manual_seed(SEED)
np.random.seed(SEED)

Device: cuda
GPU: Tesla T4
GPU Memory: 15.64 GB


In [3]:
# ============================================================
# LOAD DATASETS
# ============================================================

print("Loading datasets...")

train_df = pd.read_csv(
    os.path.join(AUGMENTED_DATASET_DIR, "train_augmented.csv")
)

val_df = pd.read_csv(
    os.path.join(AUGMENTED_DATASET_DIR, "val_augmented.csv")
)

test_clean_df = pd.read_csv(
    os.path.join(AUGMENTED_DATASET_DIR, "test_clean.csv")
)

test_augmented_df = pd.read_csv(
    os.path.join(AUGMENTED_DATASET_DIR, "test_augmented.csv")
)

print(f"Train: {len(train_df):,} rows")
print(f"Validation: {len(val_df):,} rows")
print(f"Clean test: {len(test_clean_df):,} rows")
print(f"Augmented test: {len(test_augmented_df):,} rows")


# ============================================================
# CONVERT TO BINARY LABELS
# ============================================================

# French and German are combined into "monolingual".
# Mixed sentences are classified as "code_switched".

def convert_to_binary(language):
    if language == "mixed":
        return "code_switched"
    return "monolingual"


for df in [train_df, val_df, test_clean_df, test_augmented_df]:
    df["binary_language"] = df["language"].apply(convert_to_binary)


print("\nBinary label distribution:")

print(
    f"Train monolingual: "
    f"{(train_df['binary_language'] == 'monolingual').sum():,}"
)

print(
    f"Train code-switched: "
    f"{(train_df['binary_language'] == 'code_switched').sum():,}"
)

print(
    f"Validation monolingual: "
    f"{(val_df['binary_language'] == 'monolingual').sum():,}"
)

print(
    f"Validation code-switched: "
    f"{(val_df['binary_language'] == 'code_switched').sum():,}"
)


# ============================================================
# LABEL MAPPING
# ============================================================

label2id = {
    "monolingual": 0,
    "code_switched": 1
}

id2label = {
    0: "monolingual",
    1: "code_switched"
}

num_labels = len(label2id)

print(f"\nLabel mapping: {label2id}")


# ============================================================
# LOAD MODEL AND TOKENIZER
# ============================================================

print(f"\nLoading {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
).to(device)

print(f"Model loaded and moved to {device}")
print(
    f"Total parameters: "
    f"{sum(p.numel() for p in model.parameters()):,}"
)

Loading datasets...
Train: 212,022 rows
Validation: 43,224 rows
Clean test: 39,740 rows
Augmented test: 39,740 rows

Binary label distribution:
Train monolingual: 160,020
Train code-switched: 52,002
Validation monolingual: 32,690
Validation code-switched: 10,534

Label mapping: {'monolingual': 0, 'code_switched': 1}

Loading xlm-roberta-base...


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded and moved to cuda
Total parameters: 278,045,186


In [4]:
# ============================================================
# CUSTOM DATASET CLASS
# ============================================================

class BinaryCodeSwitchingDataset(Dataset):

    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = [label2id[label] for label in labels]
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        text = self.texts[idx]
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": torch.tensor(label, dtype=torch.long)
        }


# ============================================================
# CREATE DATASETS
# ============================================================

print("Creating datasets...")

train_dataset = BinaryCodeSwitchingDataset(
    train_df["text"].values,
    train_df["binary_language"].values,
    tokenizer,
    MAX_LENGTH
)

val_dataset = BinaryCodeSwitchingDataset(
    val_df["text"].values,
    val_df["binary_language"].values,
    tokenizer,
    MAX_LENGTH
)

test_clean_dataset = BinaryCodeSwitchingDataset(
    test_clean_df["text"].values,
    test_clean_df["binary_language"].values,
    tokenizer,
    MAX_LENGTH
)

test_augmented_dataset = BinaryCodeSwitchingDataset(
    test_augmented_df["text"].values,
    test_augmented_df["binary_language"].values,
    tokenizer,
    MAX_LENGTH
)

print(f"Train dataset: {len(train_dataset):,} samples")
print(f"Validation dataset: {len(val_dataset):,} samples")
print(f"Clean test dataset: {len(test_clean_dataset):,} samples")
print(f"Augmented test dataset: {len(test_augmented_dataset):,} samples")


# ============================================================
# CREATE DATA LOADERS
# ============================================================

print("\nCreating data loaders...")

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)

test_clean_loader = DataLoader(
    test_clean_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)

test_augmented_loader = DataLoader(
    test_augmented_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)

print(f"Train loader: {len(train_loader):,} batches")
print(f"Validation loader: {len(val_loader):,} batches")
print(f"Clean test loader: {len(test_clean_loader):,} batches")
print(f"Augmented test loader: {len(test_augmented_loader):,} batches")


# ============================================================
# VERIFY DATA LOADER
# ============================================================

sample_batch = next(iter(train_loader))

print("\nTraining batch verification:")
print(f"Input IDs shape: {sample_batch['input_ids'].shape}")
print(f"Attention mask shape: {sample_batch['attention_mask'].shape}")
print(f"Labels shape: {sample_batch['labels'].shape}")
print(
    f"Unique labels: "
    f"{torch.unique(sample_batch['labels']).tolist()}"
)

Creating datasets...
Train dataset: 212,022 samples
Validation dataset: 43,224 samples
Clean test dataset: 39,740 samples
Augmented test dataset: 39,740 samples

Creating data loaders...
Train loader: 3,313 batches
Validation loader: 676 batches
Clean test loader: 621 batches
Augmented test loader: 621 batches

Training batch verification:
Input IDs shape: torch.Size([64, 128])
Attention mask shape: torch.Size([64, 128])
Labels shape: torch.Size([64])
Unique labels: [0, 1]


In [5]:
# ============================================================
# OPTIMIZER AND LEARNING-RATE SCHEDULER
# ============================================================

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

total_steps = len(train_loader) * NUM_EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=total_steps
)

print(f"Total training steps: {total_steps:,}")
print(f"Warm-up steps: {WARMUP_STEPS}")


# ============================================================
# TRAINING FUNCTION
# ============================================================

def train_epoch(model, loader, optimizer, scheduler, device):

    model.train()
    total_loss = 0
    total_correct = 0
    total_samples = 0

    for batch_idx, batch in enumerate(loader):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        # Forward pass
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        logits = outputs.logits

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()
        scheduler.step()

        # Track training metrics
        total_loss += loss.item()

        predictions = torch.argmax(logits, dim=1)

        total_correct += (
            predictions == labels
        ).sum().item()

        total_samples += labels.size(0)

        if (batch_idx + 1) % 500 == 0:

            avg_loss = total_loss / (batch_idx + 1)
            accuracy = total_correct / total_samples

            print(
                f"  Batch {batch_idx + 1}/{len(loader)} | "
                f"Loss: {avg_loss:.4f} | "
                f"Accuracy: {accuracy:.4f}"
            )

    epoch_loss = total_loss / len(loader)
    epoch_accuracy = total_correct / total_samples

    return epoch_loss, epoch_accuracy


# ============================================================
# EVALUATION FUNCTION
# ============================================================

def evaluate(model, loader, device):

    model.eval()

    total_loss = 0
    all_predictions = []
    all_labels = []

    with torch.no_grad():

        for batch in loader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            total_loss += outputs.loss.item()

            predictions = torch.argmax(
                outputs.logits,
                dim=1
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

    epoch_loss = total_loss / len(loader)

    accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            all_labels,
            all_predictions,
            average="binary",
            zero_division=0
        )
    )

    return (
        epoch_loss,
        accuracy,
        precision,
        recall,
        f1,
        all_predictions,
        all_labels
    )

Total training steps: 9,939
Warm-up steps: 500


In [ ]:
# ============================================================
# TRAINING LOOP
# ============================================================

print("\n" + "=" * 60)
print("STARTING BINARY CLASSIFIER TRAINING")
print("=" * 60)

best_val_f1 = 0
best_epoch = 0
patience = 2
patience_counter = 0

training_history = {
    "epoch": [],
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": [],
    "val_f1": []
}

for epoch in range(NUM_EPOCHS):

    print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")
    print("-" * 60)

    # Training
    train_loss, train_accuracy = train_epoch(
        model,
        train_loader,
        optimizer,
        scheduler,
        device
    )

    print(
        f"Train Loss: {train_loss:.4f} | "
        f"Train Accuracy: {train_accuracy:.4f}"
    )

    # Validation
    (
        val_loss,
        val_accuracy,
        val_precision,
        val_recall,
        val_f1,
        _,
        _
    ) = evaluate(
        model,
        val_loader,
        device
    )

    print(
        f"Val Loss: {val_loss:.4f} | "
        f"Val Accuracy: {val_accuracy:.4f} | "
        f"Val F1: {val_f1:.4f}"
    )

    # Store training history
    training_history["epoch"].append(epoch + 1)
    training_history["train_loss"].append(train_loss)
    training_history["train_accuracy"].append(train_accuracy)
    training_history["val_loss"].append(val_loss)
    training_history["val_accuracy"].append(val_accuracy)
    training_history["val_f1"].append(val_f1)

    # Save checkpoint for recovery
    checkpoint_path = os.path.join(
        MODEL_OUTPUT_DIR,
        f"checkpoint_epoch_{epoch + 1}.pt"
    )

    torch.save(
        {
            "epoch": epoch + 1,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "best_val_f1": best_val_f1,
            "training_history": training_history
        },
        checkpoint_path
    )

    # Save model if validation F1 improves
    if val_f1 > best_val_f1:

        best_val_f1 = val_f1
        best_epoch = epoch + 1
        patience_counter = 0

        model_save_path = os.path.join(
            MODEL_OUTPUT_DIR,
            "best_model.pt"
        )

        torch.save(
            model.state_dict(),
            model_save_path
        )

        print(
            f"✓ Best model saved (F1: {val_f1:.4f})"
        )

    else:

        patience_counter += 1

        if patience_counter >= patience:

            print(
                f"\nEarly stopping triggered "
                f"after {best_epoch} epochs"
            )

            break


print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)

print(
    f"Best epoch: {best_epoch} "
    f"(F1: {best_val_f1:.4f})"
)

# Save training history
history_df = pd.DataFrame(training_history)

history_path = os.path.join(
    RESULTS_OUTPUT_DIR,
    "training_history_binary.csv"
)

history_df.to_csv(
    history_path,
    index=False
)

print(f"✓ Training history saved: {history_path}")

# Load best model for final evaluation
model.load_state_dict(
    torch.load(
        os.path.join(
            MODEL_OUTPUT_DIR,
            "best_model.pt"
        )
    )
)

print("✓ Best model loaded for evaluation")


STARTING TRAINING (BINARY CLASSIFIER)

Epoch 1/3
------------------------------------------------------------
  Batch 500/6051 | Loss: 0.3599 | Accuracy: 0.8683
  Batch 1000/6051 | Loss: 0.2505 | Accuracy: 0.9167
  Batch 1500/6051 | Loss: 0.1989 | Accuracy: 0.9368
  Batch 2000/6051 | Loss: 0.1674 | Accuracy: 0.9481
  Batch 2500/6051 | Loss: 0.1466 | Accuracy: 0.9554
  Batch 3000/6051 | Loss: 0.1306 | Accuracy: 0.9607
  Batch 3500/6051 | Loss: 0.1187 | Accuracy: 0.9645
  Batch 4000/6051 | Loss: 0.1090 | Accuracy: 0.9677
  Batch 4500/6051 | Loss: 0.1010 | Accuracy: 0.9703
  Batch 5000/6051 | Loss: 0.0944 | Accuracy: 0.9725
  Batch 5500/6051 | Loss: 0.0890 | Accuracy: 0.9742
  Batch 6000/6051 | Loss: 0.0844 | Accuracy: 0.9757
Train Loss: 0.0839 | Train Accuracy: 0.9758
Val Loss: 0.0258 | Val Accuracy: 0.9941 | Val F1: 0.9829
✓ Checkpoint saved: epoch 1
✓ Best model saved (F1: 0.9829)

Epoch 2/3
------------------------------------------------------------
  Batch 500/6051 | Loss: 0.0260 |

In [6]:
# ============================================================
# FINAL MODEL EVALUATION
# ============================================================

print("\n" + "=" * 60)
print("FINAL MODEL EVALUATION")
print("=" * 60)

# Load the saved best model
checkpoint_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "best_model.pt"
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

model.load_state_dict(
    torch.load(
        checkpoint_path,
        map_location=device
    )
)

model = model.to(device)
model.eval()

print("Saved model loaded successfully.")


# ============================================================
# CLEAN TEST SET
# ============================================================

print("\n" + "-" * 60)
print("CLEAN TEST SET")
print("-" * 60)

(
    test_clean_loss,
    test_clean_acc,
    test_clean_prec,
    test_clean_rec,
    test_clean_f1,
    _,
    _
) = evaluate(
    model,
    test_clean_loader,
    device
)

print(f"Test samples: {len(test_clean_df):,}")
print(f"Loss:         {test_clean_loss:.4f}")
print(f"Accuracy:     {test_clean_acc:.4f}")
print(f"Precision:    {test_clean_prec:.4f}")
print(f"Recall:       {test_clean_rec:.4f}")
print(f"F1 Score:     {test_clean_f1:.4f}")


# ============================================================
# AUGMENTED TEST SET
# ============================================================

print("\n" + "-" * 60)
print("AUGMENTED TEST SET")
print("-" * 60)

(
    test_aug_loss,
    test_aug_acc,
    test_aug_prec,
    test_aug_rec,
    test_aug_f1,
    _,
    _
) = evaluate(
    model,
    test_augmented_loader,
    device
)

print(f"Test samples: {len(test_augmented_df):,}")
print(f"Loss:         {test_aug_loss:.4f}")
print(f"Accuracy:     {test_aug_acc:.4f}")
print(f"Precision:    {test_aug_prec:.4f}")
print(f"Recall:       {test_aug_rec:.4f}")
print(f"F1 Score:     {test_aug_f1:.4f}")


# ============================================================
# CLEAN VS AUGMENTED
# ============================================================

f1_difference = test_clean_f1 - test_aug_f1

print("\n" + "-" * 60)
print("CLEAN VS AUGMENTED")
print("-" * 60)

print(f"Clean test F1:     {test_clean_f1:.4f}")
print(f"Augmented test F1: {test_aug_f1:.4f}")
print(f"F1 difference:     {f1_difference:+.4f}")

print("\n" + "=" * 60)
print("EVALUATION COMPLETE")
print("=" * 60)


FINAL MODEL EVALUATION


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Saved model loaded successfully.

------------------------------------------------------------
CLEAN TEST SET
------------------------------------------------------------
Test samples: 39,740
Loss:         0.0437
Accuracy:     0.9911
Precision:    0.9794
Recall:       0.9845
F1 Score:     0.9819

------------------------------------------------------------
AUGMENTED TEST SET
------------------------------------------------------------
Test samples: 39,740
Loss:         0.2452
Accuracy:     0.9550
Precision:    0.8819
Recall:       0.9425
F1 Score:     0.9112

------------------------------------------------------------
CLEAN VS AUGMENTED
------------------------------------------------------------
Clean test F1:     0.9819
Augmented test F1: 0.9112
F1 difference:     +0.0707

EVALUATION COMPLETE
